# Hypothesis Testing
**Project:** Impact of Global Crises on Financial Markets

We test whether global crises have a **statistically significant** effect on market returns.

## Hypotheses
- **H₀**: Global crises do NOT have statistically significant effects on market returns or sectoral behavior.
- **H₁**: Global crises DO have statistically significant effects on market returns and sectoral behavior, with different sectors responding in systematically different directions depending on the type of crisis.

## Tests Used
| Test | Type | Purpose |
|---|---|---|
| Mann-Whitney U | Non-parametric | Compare pre vs post returns — no normality assumption |
| Kruskal-Wallis | Non-parametric | Compare post-crisis returns across all 6 crises simultaneously |
| Interrupted Time Series (ITS) | Regression-based | Model pre-crisis trend and measure deviation after crisis — captures both immediate level change and slope change |

> **Note**: Daily financial returns have fat tails and non-normal distributions. Mann-Whitney U and Kruskal-Wallis are primary tests. T-test results are shown for reference only. 


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import stats
from scipy.stats import f_oneway
!pip install statsmodels

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries imported.')

In [ ]:
# Load data
save_dir = os.path.expanduser('~/Desktop/data')
returns = pd.read_csv(os.path.join(save_dir, 'returns.csv'), index_col=0, parse_dates=True)

# Crisis dates
CRISES = {
    '9/11 (2001)':             '2001-09-11',
    'Iraq War (2003)':         '2003-03-20',
    'Financial Crisis (2008)': '2008-09-15',
    'COVID-19 (2020)':         '2020-03-11',
    'Russia-Ukraine (2022)':   '2022-02-24',
    'Israel-Hamas (2023)':     '2023-10-07',
}

print(f'Returns loaded: {returns.shape[0]} days, {returns.shape[1]} assets')

## 1. Two-Sample T-Test + Mann-Whitney U

For each crisis, we compare S&P 500 daily returns in the **30 days before** vs **30 days after**.

- **Two-sample t-test**: parametric, shown for reference
- **Mann-Whitney U**: non-parametric, primary test — does not assume normality

> *Note: Daily financial returns are known to have fat tails and non-normal distributions. Mann-Whitney U is the primary test as it does not assume normality. T-test results are shown for reference only.*

**Decision rule**: p < 0.05 → reject H₀ → crisis had a significant effect

In [ ]:
results = []

for crisis_name, crisis_date_str in CRISES.items():
    crisis_date = pd.Timestamp(crisis_date_str)
    pre_start = crisis_date - pd.Timedelta(days=30)
    post_end  = crisis_date + pd.offsets.BusinessDay(30)

    pre  = returns['SP500'].loc[pre_start  : crisis_date - pd.Timedelta(days=1)].dropna()
    post = returns['SP500'].loc[crisis_date : post_end].dropna()

    if len(pre) < 5 or len(post) < 5:
        continue

    # Two-sample t-test (reference only)
    t_stat, t_pval = stats.ttest_ind(pre, post)

    # Mann-Whitney U (primary non-parametric test)
    u_stat, u_pval = stats.mannwhitneyu(pre, post, alternative='two-sided')

    results.append({
        'Crisis':       crisis_name,
        'Pre mean (%)': round(pre.mean(), 3),
        'Post mean (%)':round(post.mean(), 3),
        'T-test p':     round(t_pval, 4),
        'MWU p':        round(u_pval, 4),
        'Significant?': 'YES *' if u_pval < 0.05 else 'NO'
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## 2. Window Size Comparison (7, 30, 60 days)

We repeat the Mann-Whitney U test with three different window sizes to check whether the choice of window affects the results.

In [ ]:
window_results = []

for window in [7, 30, 60]:
    for crisis_name, crisis_date_str in CRISES.items():
        crisis_date = pd.Timestamp(crisis_date_str)
        pre_start = crisis_date - pd.offsets.BusinessDay(window)
        post_end  = crisis_date + pd.offsets.BusinessDay(window)

        pre  = returns['SP500'].loc[pre_start  : crisis_date - pd.Timedelta(days=1)].dropna()
        post = returns['SP500'].loc[crisis_date : post_end].dropna()

        if len(pre) < 5 or len(post) < 5:
            continue

        u_stat, u_pval = stats.mannwhitneyu(pre, post, alternative='two-sided')

        window_results.append({
            'Window':       f'{window}d',
            'Crisis':       crisis_name,
            'Pre mean (%)': round(pre.mean(), 3),
            'Post mean (%)':round(post.mean(), 3),
            'MWU p':        round(u_pval, 4),
            'Significant?': 'YES *' if u_pval < 0.05 else 'NO'
        })

window_df = pd.DataFrame(window_results)
print(window_df.to_string(index=False))

In [ ]:
# Pivot MWU p-values by window size
pivot_window = window_df.pivot(index='Crisis', columns='Window', values='MWU p')

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot_window, annot=True, fmt='.3f', cmap='RdYlGn_r',
            vmin=0, vmax=0.1, linewidths=0.5, ax=ax)
ax.set_title('Mann-Whitney U p-values by Window Size\n(Red = significant p < 0.05, Green = not significant)')
plt.tight_layout()
plt.show()

## 3. P-Value Chart 

In [ ]:
# P-value chart for all window sizes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, window in enumerate([7, 30, 60]):
    ax = axes[idx]
    w_data = window_df[window_df['Window'] == f'{window}d']
    
    colors = ['#d62728' if p < 0.05 else '#2196F3' for p in w_data['MWU p']]
    ax.bar(range(len(w_data)), w_data['MWU p'], color=colors, alpha=0.8)
    ax.axhline(y=0.05, color='red', linestyle='--', linewidth=1.5, label='p = 0.05')
    ax.set_xticks(range(len(w_data)))
    ax.set_xticklabels(w_data['Crisis'], rotation=25, ha='right', fontsize=8)
    ax.set_ylabel('MWU p-value')
    ax.set_title(f'{window}-day window')
    ax.set_ylim(0, 1.05)
    ax.legend()

plt.suptitle('Mann-Whitney U p-values: Pre vs Post Crisis S&P 500 Returns\n(Red bars = significant p < 0.05)', 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Distribution of Returns: Pre vs Post

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, (crisis_name, crisis_date_str) in enumerate(CRISES.items()):
    crisis_date = pd.Timestamp(crisis_date_str)
    pre_start   = crisis_date - pd.Timedelta(days=30)
    post_end    = crisis_date + pd.offsets.BusinessDay(30)

    pre  = returns['SP500'].loc[pre_start  : crisis_date - pd.Timedelta(days=1)].dropna()
    post = returns['SP500'].loc[crisis_date : post_end].dropna()

    ax = axes[i]
    ax.hist(pre,  bins=12, alpha=0.6, color='#2196F3', label='Pre-crisis')
    ax.hist(post, bins=12, alpha=0.6, color='#F44336', label='Post-crisis')
    ax.axvline(pre.mean(),  color='#2196F3', linestyle='--', linewidth=1.5)
    ax.axvline(post.mean(), color='#F44336', linestyle='--', linewidth=1.5)
    ax.set_title(crisis_name, fontsize=10)
    ax.set_xlabel('Daily Return (%)')
    ax.legend(fontsize=7)

plt.suptitle('Distribution of S&P 500 Daily Returns: Pre vs Post Crisis', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. Sector-Level Tests

Do different sectors respond differently to crises?

We apply Mann-Whitney U test for each sector in each crisis where data is available.

In [ ]:
sectors = ['SP500', 'Defense', 'Energy', 'Health', 'Tech', 'Aviation']

sector_results = []

for crisis_name, crisis_date_str in CRISES.items():
    crisis_date = pd.Timestamp(crisis_date_str)
    pre_start   = crisis_date - pd.Timedelta(days=30)
    post_end    = crisis_date + pd.offsets.BusinessDay(30)

    for sector in sectors:
        if sector not in returns.columns:
            continue

        pre  = returns[sector].loc[pre_start  : crisis_date - pd.Timedelta(days=1)].dropna()
        post = returns[sector].loc[crisis_date : post_end].dropna()

        if len(pre) < 5 or len(post) < 5:
            continue

        # Primary: Mann-Whitney U
        u_stat, u_pval = stats.mannwhitneyu(pre, post, alternative='two-sided')
        # Reference: T-test
        t_stat, t_pval = stats.ttest_ind(pre, post)

        sector_results.append({
            'Crisis':       crisis_name,
            'Sector':       sector,
            'Pre mean (%)': round(pre.mean(), 3),
            'Post mean (%)':round(post.mean(), 3),
            'T-test p':     round(t_pval, 4),
            'MWU p':        round(u_pval, 4),
            'Significant?': 'YES *' if u_pval < 0.05 else 'NO'
        })

sector_df = pd.DataFrame(sector_results)
print(sector_df.to_string(index=False))

In [ ]:
# Heatmap of MWU p-values by sector and crisis
pivot = sector_df.pivot(index='Crisis', columns='Sector', values='MWU p')

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn_r',
            vmin=0, vmax=0.1, linewidths=0.5, ax=ax)
ax.set_title('Mann-Whitney U p-values by Sector and Crisis\n(Red = significant p < 0.05, Green = not significant)')
plt.tight_layout()
plt.show()

## 6. MSCI World vs S&P 500 Post-Crisis

In [ ]:
if 'MSCI_World' not in returns.columns:
    print('MSCI World not found in returns data.')
else:
    recent_crises = {
        'COVID-19 (2020)':       '2020-03-11',
        'Russia-Ukraine (2022)': '2022-02-24',
        'Israel-Hamas (2023)':   '2023-10-07',
    }

    msci_results = []

    for crisis_name, crisis_date_str in recent_crises.items():
        crisis_date = pd.Timestamp(crisis_date_str)
        post_end    = crisis_date + pd.offsets.BusinessDay(30)

        sp500_post = returns['SP500'].loc[crisis_date : post_end].dropna()
        msci_post  = returns['MSCI_World'].loc[crisis_date : post_end].dropna()

        if len(sp500_post) < 5 or len(msci_post) < 5:
            continue

        t_stat, t_pval = stats.ttest_ind(sp500_post, msci_post)
        u_stat, u_pval = stats.mannwhitneyu(sp500_post, msci_post, alternative='two-sided')

        msci_results.append({
            'Crisis':         crisis_name,
            'SP500 mean (%)': round(sp500_post.mean(), 3),
            'MSCI mean (%)':  round(msci_post.mean(), 3),
            'T-test p':       round(t_pval, 4),
            'MWU p':          round(u_pval, 4),
            'Significant?':   'YES *' if u_pval < 0.05 else 'NO'
        })

    msci_df = pd.DataFrame(msci_results)
    print(msci_df.to_string(index=False))

## 7. Kruskal-Wallis Test

Are post-crisis returns significantly different across all 6 crises?

Kruskal-Wallis is the non-parametric alternative to ANOVA — no normality assumption required.

In [ ]:
post_groups = []
group_names = []

for crisis_name, crisis_date_str in CRISES.items():
    crisis_date = pd.Timestamp(crisis_date_str)
    post_end    = crisis_date + pd.offsets.BusinessDay(30)
    post = returns['SP500'].loc[crisis_date : post_end].dropna()
    if len(post) >= 5:
        post_groups.append(post.values)
        group_names.append(crisis_name)

kw_stat, kw_pval = stats.kruskal(*post_groups)

print('Kruskal-Wallis Test: Are post-crisis returns different across all 6 crises?')
print(f'  H-statistic: {kw_stat:.4f}')
print(f'  p-value:     {kw_pval:.4f}')
if kw_pval < 0.05:
    print('  Result: REJECT H₀ — post-crisis returns differ significantly across crises (p < 0.05)')
else:
    print('  Result: FAIL TO REJECT H₀ — cannot prove returns differ across crises (p >= 0.05)')

## 8. Interrupted Time Series (ITS) Analysis

Interrupted Time Series modeling asks: **"What would the S&P 500 trend have been without the crisis?"**

Unlike t-tests that only compare means, ITS models the pre-crisis trend and measures how much the post-crisis reality deviated from that expected trend.

For each crisis we fit the model:

$$Y_t = \beta_0 + \beta_1 \cdot t + \beta_2 \cdot D_t + \beta_3 \cdot (t - T_0) \cdot D_t + \epsilon_t$$

Where:
- $t$ = time index
- $D_t$ = 0 before crisis, 1 after crisis
- $\beta_2$ = **level change** at crisis date (immediate impact)
- $\beta_3$ = **slope change** after crisis (change in trend)

In [ ]:
import statsmodels.api as sm

def run_its(crisis_name, crisis_date_str, window_days=60):
    """
    Fit an ITS model for a single crisis.
    Returns: fitted model, cumulative series, counterfactual, crisis_date
    """
    crisis_date = pd.Timestamp(crisis_date_str)
    start = crisis_date - pd.offsets.BusinessDay(window_days)
    end   = crisis_date + pd.offsets.BusinessDay(window_days)

    # Get SP500 returns and convert to cumulative index (starts at 100)
    series     = returns['SP500'].loc[start:end].dropna()
    cum_series = (1 + series / 100).cumprod() * 100

    # Build ITS variables
    n          = len(cum_series)
    t          = np.arange(n)                                        # time index
    D          = np.array(cum_series.index >= crisis_date, dtype=int) # 0=pre, 1=post
    crisis_idx = np.argmax(D)                                        # index of crisis
    t_post     = np.where(D == 1, t - crisis_idx, 0)                # time since crisis

    # Design matrix
    X = pd.DataFrame({'const': 1, 't': t, 'D': D, 't_post': t_post})
    y = cum_series.values

    # Fit OLS with HAC (Newey-West) standard errors
    # HAC corrects for BOTH heteroskedasticity AND autocorrelation —
    # essential for time series, since residuals on consecutive days are correlated.
    # maxlags=5: rule of thumb is T^(1/3) ≈ 5 for our 121-day window.
    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})

    # Counterfactual: what would trend look like without the crisis?
    X_counter            = X.copy()
    X_counter['D']       = 0
    X_counter['t_post']  = 0
    counterfactual = model.predict(X_counter)

    return model, cum_series, counterfactual, crisis_date

In [ ]:
# ── Plot ITS for all 6 crises ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

its_summary = []

for i, (crisis_name, crisis_date_str) in enumerate(CRISES.items()):
    try:
        model, cum_series, counterfactual, crisis_date = run_its(
            crisis_name, crisis_date_str, window_days=60
        )

        ax     = axes[i]
        dates  = cum_series.index
        actual = cum_series.values

        # Plot actual vs counterfactual
        ax.plot(dates, actual,         color='#2196F3', linewidth=1.5, label='Actual')
        ax.plot(dates, counterfactual, color='#FF9800', linewidth=1.5,
                linestyle='--', label='Counterfactual')
        ax.axvline(crisis_date, color='red', linewidth=1.2,
                   linestyle='--', label='Crisis date')

        # Shade the gap between actual and counterfactual post-crisis
        post_mask = dates >= crisis_date
        ax.fill_between(
            dates[post_mask],
            actual[post_mask],
            counterfactual[post_mask],
            alpha=0.2, color='red', label='Impact'
        )

        ax.set_title(crisis_name, fontsize=10)
        ax.set_ylabel('Cumulative Return (indexed to 100)')
        ax.legend(fontsize=7)
        ax.tick_params(axis='x', rotation=20)

        # Collect results
        params = model.params
        pvals  = model.pvalues
        its_summary.append({
            'Crisis':       crisis_name,
            'level_change': round(params['D'], 3),
            'b2_pvalue':    round(pvals['D'], 4),
            'slope_change': round(params['t_post'], 3),
            'b3_pvalue':    round(pvals['t_post'], 4),
            'Level sig?':   'YES *' if pvals['D'] < 0.05 else 'NO',
            'Slope sig?':   'YES *' if pvals['t_post'] < 0.05 else 'NO',
            'DW': round(sm.stats.stattools.durbin_watson(model.resid), 2),
        })

    except Exception as e:
        print(f'{crisis_name}: {e}')

plt.suptitle('Interrupted Time Series: Actual vs Counterfactual S&P 500\n'
             '(±60 business days around each crisis)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── ITS Results Table ────────────────────────────────────────────────────────
its_df = pd.DataFrame(its_summary)
print('ITS Results:')
print(its_df.to_string(index=False))

In [ ]:
# ── Visualize Level and Slope Changes ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Level change (β₂) — immediate impact at crisis date
colors_level = ['#d62728' if p < 0.05 else '#90CAF9' for p in its_df['b2_pvalue']]
axes[0].barh(its_df['Crisis'], its_df['level_change'], color=colors_level)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('ITS: Immediate Level Change at Crisis Date\n(Red = significant p < 0.05)')
axes[0].set_xlabel('Level change (indexed)')

# Slope change (β₃) — change in trend after crisis
colors_slope = ['#d62728' if p < 0.05 else '#90CAF9' for p in its_df['b3_pvalue']]
axes[1].barh(its_df['Crisis'], its_df['slope_change'], color=colors_slope)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('ITS: Slope Change After Crisis\n(Red = significant p < 0.05)')
axes[1].set_xlabel('Daily slope change (indexed)')

plt.tight_layout()
plt.show()

## Summary of Hypothesis Test Results

> **Note on test selection**: Daily financial returns exhibit fat tails and non-normal distributions. 
> Mann-Whitney U (two groups) and Kruskal-Wallis (multiple groups) are the primary tests. 
> T-test results are shown for reference only.

### SP500 — Pre vs Post (30-day window, Mann-Whitney U)
| Crisis | Pre Mean | Post Mean | MWU p | Significant? |
|---|---|---|---|---|
| 9/11 (2001) | -0.421 | -0.010 | 0.3006 | NO |
| Iraq War (2003) | 0.220 | 0.166 | 0.9631 | NO |
| Financial Crisis (2008) | -0.181 | -1.144 | 0.1501 | NO |
| COVID-19 (2020) | -0.639 | 0.024 | 0.5594 | NO |
| Russia-Ukraine (2022) | -0.194 | 0.213 | 0.3707 | NO |
| Israel-Hamas (2023) | -0.160 | 0.160 | 0.1854 | NO |

### Window Size Comparison (MWU p-values)
| Crisis | 7-day | 30-day | 60-day |
|---|---|---|---|
| 9/11 (2001) | 0.247 | 0.384 | 0.235 |
| Iraq War (2003) | 0.152 | 0.699 | 0.133 |
| Financial Crisis (2008) | 0.694 | 0.078 | 0.479 |
| COVID-19 (2020) | 0.463 | 0.677 | 0.213 |
| Russia-Ukraine (2022) | 0.491 | 0.100 | 0.796 |
| Israel-Hamas (2023) | 1.000 | 0.292 | 0.063 |

### Kruskal-Wallis
| Test | Statistic | p-value | Result |
|---|---|---|---|
| Kruskal-Wallis | H = 5.4885 | 0.3592 | Fail to reject H₀ |

### Interrupted Time Series (ITS) — ±60 business days

Standard errors computed using the Newey-West (HAC) estimator with 5 lags to correct for both heteroskedasticity and autocorrelation in the residuals.

| Crisis | Level Change (β₂) | β₂ p-value | Slope Change (β₃) | β₃ p-value | Level sig? | Slope sig? | DW |
|---|---|---|---|---|---|---|---|
| 9/11 (2001) | -9.981 | 0.0000 | +0.347 | 0.0000 | YES * | YES * | 0.77 |
| Iraq War (2003) | +4.694 | 0.0002 | +0.437 | 0.0000 | YES * | YES * | 0.41 |
| Financial Crisis (2008) | -7.244 | 0.0127 | -0.483 | 0.0000 | YES * | YES * | 0.48 |
| COVID-19 (2020) | -21.538 | 0.0000 | +0.444 | 0.0000 | YES * | YES * | 0.72 |
| Russia-Ukraine (2022) | +1.890 | 0.3735 | +0.006 | 0.9297 | NO | NO | 0.21 |
| Israel-Hamas (2023) | -2.609 | 0.1039 | +0.326 | 0.0000 | NO | YES * | 0.26 |

### Interpretation
- **MWU tests fail to reject H₀** — 30-day window has only ~20 trading days, insufficient power to detect mean differences against high daily volatility
- **ITS strongly supports H₁** — 5 out of 6 crises show statistically significant level and/or slope changes (p < 0.05)
- **COVID-19** had the largest immediate impact: −21.5 index points at onset, followed by recovery (+0.444 daily slope)
- **Financial Crisis (2008)** is the only crisis with a negative slope change (−0.483) — market continued deteriorating post-crisis rather than recovering
- **Israel-Hamas (2023)** shows no significant immediate level change (p = 0.10), but a significant positive slope change — suggesting markets absorbed the shock gradually rather than as an immediate impact, consistent with regional rather than global pricing
- **Russia-Ukraine (2022)** is the only crisis where ITS found no significant effect on either level or slope — consistent with EDA showing markets quickly priced in the geopolitical risk
- **Durbin-Watson statistics (0.21–0.77)** confirm strong positive autocorrelation in residuals, validating the use of HAC standard errors over conventional OLS
- **Conclusion**: H₁ is supported by ITS analysis. Crises significantly alter the level and/or trajectory of market returns. MWU tests are underpowered for this context — ITS is a more appropriate method for event impact analysis.